# Lab 6 — MLflow Experiment Log

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Goals

1. Train a KNN pipeline and compute **test accuracy**.
2. Log **parameters**, **metrics**, and the **model** to MLflow.
3. Persist runs in a local **SQLite** tracking store.
4. Write `metrics.json` and **track it with DVC** (`dvc init`, `dvc add`, `dvc status`).

> **Quick check:** accuracy ≈ **0.58** · k = **7** · `metrics.json` written · `output/metrics.json.dvc` created




## MLflow tracking in one slide

| Artifact | What gets logged |
|----------|------------------|
| **Parameters** | `model`, `k` — reproducible config |
| **Metrics** | `accuracy` — compare runs in the UI |
| **Model** | Serialized sklearn `Pipeline` — deploy or audit later |
| **metrics.json** | Flat file for **DVC** (`dvc add`) in the classroom demo |

```text
train → MLflow log → metrics.json → dvc add → metrics.json.dvc pointer
```

Lab 4 exposed the model via FastAPI; Lab 6 **versions** the training run.

---

## 1. Load data and train KNN (k = 7)

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

OUTPUT_DIR = GH_ROOT / "hands-on" / "04-distance-mlops" / "output"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

k = 7
model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ]
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"k: {k}")
print(f"test accuracy: {accuracy:.4f}")



This lab uses **k=7** (accuracy ≈ 0.58 on this split) — a deliberate choice to show logging a specific run, not necessarily the Lab 3 optimum (k=3).

### 1b. Confusion matrix preview

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)


---

## 2. Configure MLflow tracking

In [ ]:
import mlflow

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
mlflow_db = OUTPUT_DIR / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db.as_posix()}")
mlflow.set_experiment("cisco-aiml-day04-lending-club")

print(f"tracking URI: sqlite:///{mlflow_db.name}")
print(f"experiment: cisco-aiml-day04-lending-club")


### 2b. Tracking URI explained

In [ ]:
print(f"SQLite DB path: {mlflow_db}")
print(f"DB exists before run: {mlflow_db.is_file()}")


---

## 3. Log the run

In [ ]:
with mlflow.start_run(run_name="knn-baseline") as run:
    mlflow.log_param("model", "KNeighborsClassifier")
    mlflow.log_param("k", k)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        skops_trusted_types=[
            "sklearn.metrics._dist_metrics.EuclideanDistance64",
            "sklearn.neighbors._kd_tree.KDTree",
        ],
    )
    run_id = run.info.run_id

print("Lab 6 — MLflow experiment log")
print(f"run_id: {run_id}")
print(f"accuracy logged: {accuracy:.4f}")


### 3b. What was logged

In [ ]:
print("Parameters: model=KNeighborsClassifier, k=7")
print(f"Metric: accuracy={accuracy:.4f}")
print(f"Artifact: model (sklearn Pipeline)")


---

## 4. Write metrics.json (DVC demo artifact)

In [ ]:
import json

from IPython.display import display

metrics_path = OUTPUT_DIR / "metrics.json"
metrics_payload = {
    "accuracy": round(accuracy, 4),
    "k": k,
    "run_id": run_id,
}
metrics_path.write_text(json.dumps(metrics_payload), encoding="utf-8")

print(f"metrics artifact (DVC demo): {metrics_path.name}")
print(f"full path: {metrics_path}")
display(pd.DataFrame([metrics_payload]))


---

## 5. Track metrics.json with DVC

**DVC** (Data Version Control) stores large files outside Git and keeps a small `.dvc` pointer in the repo.

| Step | Command | Purpose |
|------|---------|--------|
| Init | `dvc init --no-scm` | Enable DVC in `04-distance-mlops/` (classroom; no Git required) |
| Add | `dvc add output/metrics.json` | Snapshot metrics + create `metrics.json.dvc` |
| Status | `dvc status` | Confirm workspace is clean |

MLflow logs **experiments**; DVC versions **artifact files** — complementary tools (Day 1 Lab 5).

In [ ]:
DAY_DIR = GH_ROOT / "hands-on" / "04-distance-mlops"
METRICS_REL = "output/metrics.json"

import subprocess
import sys

DVC = [sys.executable, "-m", "dvc"]

if not (DAY_DIR / ".dvc").is_dir():
    proc = subprocess.run(
        [*DVC, "init", "--no-scm"],
        cwd=DAY_DIR,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr or proc.stdout)
    print("dvc init: OK")

proc = subprocess.run(
    [*DVC, "add", METRICS_REL],
    cwd=DAY_DIR,
    capture_output=True,
    text=True,
)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr or proc.stdout)
print(proc.stdout or "dvc add: OK")

status = subprocess.run([*DVC, "status"], cwd=DAY_DIR, capture_output=True, text=True)
print(status.stdout or status.stderr)

dvc_pointer = DAY_DIR / f"{METRICS_REL}.dvc"
print(f"DVC pointer: {dvc_pointer.name}")
assert dvc_pointer.is_file(), "metrics.json.dvc should exist"


### 5b. Inspect `.dvc` pointer file

In [ ]:
dvc_content = (DAY_DIR / "output/metrics.json.dvc").read_text(encoding="utf-8")
print(dvc_content[:300])


---

## 6. Verify the metrics file

In [ ]:
loaded = json.loads(metrics_path.read_text(encoding="utf-8"))
print(loaded)
assert metrics_path.is_file()
assert loaded["k"] == k
assert loaded["run_id"] == run_id


---

## 7. Launch MLflow UI (instructor demo)

From `hands-on/04-distance-mlops/output`:

```bash
mlflow ui --backend-store-uri sqlite:///output/mlflow.db
```

Open **http://127.0.0.1:5000** — compare parameters, metrics, and the saved model artifact.

After logging runs, start the UI from `output/`:

```bash
cd hands-on/04-distance-mlops/output
mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```


---

## 8. Day 04 recap

In [ ]:
recap = pd.DataFrame({
    "lab": ["1 Distance", "2 KNN k=5", "3 Choose k", "4 FastAPI", "5 FeatureTools", "6 MLflow"],
    "checkpoint": [
        "cosine ≈ 0.90",
        "acc ≈ 0.55",
        "best k=3, acc ≈ 0.59",
        "GET/POST 200",
        "shape (1000, 6)",
        f"acc ≈ {accuracy:.2f}, metrics.json",
    ],
})
display(recap)


### 8b. MLflow vs DVC roles

| Tool | Tracks | Classroom artifact |
|------|--------|--------------------|
| **MLflow** | Experiments, params, metrics, models | `mlflow.db` |
| **DVC** | File versions via content hashes | `metrics.json.dvc` |

### 8c. Reproducibility checklist

In [ ]:
checklist = pd.DataFrame({
    "item": ["random_state=42", "k=7", "1000-row sample", "stratified split"],
    "value": ["42", str(k), str(len(df)), "yes"],
})
display(checklist)


---

## 9. Query MLflow runs (optional)

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
exp = client.get_experiment_by_name("cisco-aiml-day04-lending-club")
if exp:
    runs = client.search_runs(experiment_ids=[exp.experiment_id], max_results=3)
    for r in runs:
        print(r.info.run_name, r.data.params, r.data.metrics)


### 9b. metrics.json schema

In [ ]:
print("Required keys:", sorted(metrics_payload.keys()))
print("Sample:", metrics_payload)


### 9c. DVC status interpretation

In [ ]:
status2 = subprocess.run([*DVC, "status"], cwd=DAY_DIR, capture_output=True, text=True)
print(status2.stdout or "Data and pipelines are up to date.")


### 9d. Model artifact path in MLflow

In [ ]:
print(f"Run ID for UI lookup: {run_id}")
print(f"SQLite tracking store: {mlflow_db}")


### 9e. Compare k=3 vs k=7 accuracy

In [ ]:
pipe3 = Pipeline([("scale", StandardScaler()), ("knn", KNeighborsClassifier(3))])
pipe3.fit(X_train, y_train)
acc3 = accuracy_score(y_test, pipe3.predict(X_test))
print(f"k=3 test accuracy: {acc3:.4f}")
print(f"k=7 test accuracy: {accuracy:.4f}")
print("Lab 6 logs k=7 deliberately — compare runs in MLflow UI.")


### 9f. metrics.json for CI/CD

Teams often gate merges on `metrics.json` thresholds in CI — DVC tracks the file; MLflow tracks the experiment lineage.

### 9g. Train set size logged

In [ ]:
print(f"training rows: {len(X_train)}, test rows: {len(X_test)}")
print(f"features: {NUMERIC_FEATURES}")


### 9h. MLflow experiment tags (concept)

Add tags like `dataset=lending_club_sample` and `day=04` to filter runs in the UI when comparing across training days.

### 9i. Artifact directory layout

In [ ]:
print("Output dir contents:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}")


### 9j. DVC cache location

In [ ]:
dvc_dir = DAY_DIR / ".dvc"
print(f"DVC metadata dir exists: {dvc_dir.is_dir()}")


---

## 10. Checkpoint summary

In [ ]:
assert k == 7
assert abs(accuracy - 0.58) < 0.02
assert metrics_path.is_file()
assert loaded["accuracy"] == round(accuracy, 4)
print(f"tracking db: {mlflow_db.name}")
assert (DAY_DIR / "output/metrics.json.dvc").is_file()
print("Numbers match — you're good.")



### 10b. Day 04 MLOps stack recap

| Layer | Lab | Tool |
|-------|-----|------|
| Model | 2–3 | sklearn KNN |
| Serving | 4 | FastAPI |
| Features | 5 | FeatureTools |
| Tracking | 6 | MLflow + DVC |

## Extension — log a second run (k=3) and compare

In [ ]:
import mlflow
import mlflow.sklearn

for k_try in [3, 7]:
    pipe_k = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k_try)),
    ])
    pipe_k.fit(X_train, y_train)
    acc_k = accuracy_score(y_test, pipe_k.predict(X_test))
    with mlflow.start_run(run_name=f"knn-k{k_try}"):
        mlflow.log_param("k", k_try)
        mlflow.log_metric("accuracy", acc_k)
        print(f"logged k={k_try} accuracy={acc_k:.3f}")
print("Open MLflow UI at http://127.0.0.1:5000 to compare runs")


---

## Reflection

1. What would you log additionally for a fraud model (Day 6)?
2. Why keep both MLflow **and** a flat `metrics.json` for DVC?
3. How would you promote the logged model to the Lab 4 FastAPI service?
